# Ridge, Lasso, and Elastic Net Regression

- **Dataset split**: 80% training, 20% testing
- **Same random seed used across all models** (random_state=42)
- **Hyperparameters selected using cross-validation on the training set**
- **Final evaluation performed only on the test set**
- **Classification threshold**: 0.5

In [89]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, mean_squared_error, r2_score
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [90]:
# Load data
df = pd.read_excel("Project Database.xlsx")
df.columns = df.columns.str.strip()

# Define target variable
target = "Personal.Loan"

# Define X and Y
Y = df[target]
X = df.drop(columns=[target, "City"] )

# Convert categorical columns to dummies
X = pd.get_dummies(X, drop_first=True)

# Combine and drop missing values
data = pd.concat([Y, X], axis=1).dropna()
Y = data[target]
X = data.drop(columns=[target])

# Force everything to numeric
X = X.apply(pd.to_numeric, errors="coerce")
Y = pd.to_numeric(Y, errors="coerce")

# Drop rows made missing by coercion
data = pd.concat([Y, X], axis=1).dropna()
Y = data[target]
X = data.drop(columns=[target])

X = X.astype(float)
Y = Y.astype(float)

print("Dataset loaded.")
print(f"Total samples: {len(Y)}")
print(f"Features: {X.shape[1]}")
print(f"Target distribution: {Y.value_counts().to_dict()}")

Dataset loaded.
Total samples: 5000
Features: 12
Target distribution: {0.0: 4520, 1.0: 480}


In [91]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.20,
    random_state=42,
    stratify=Y
)

print(f"Train size: {X_train.shape[0]}")
print(f"Test size: {X_test.shape[0]}")
print(f"Train class 0: {sum(y_train == 0)}")
print(f"Train class 1: {sum(y_train == 1)}")
print(f"Test class 0: {sum(y_test == 0)}")
print(f"Test class 1: {sum(y_test == 1)}")

Train size: 4000
Test size: 1000
Train class 0: 3616
Train class 1: 384
Test class 0: 904
Test class 1: 96


In [92]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Standardization done.")

Standardization done.


In [93]:
# Ridge Regression
ridge_alphas = [0.001, 0.01, 0.1, 1, 10, 100]
ridge = Ridge()
ridge_grid = GridSearchCV(
    ridge,
    param_grid={'alpha': ridge_alphas},
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
ridge_grid.fit(X_train_scaled, y_train)
best_ridge_alpha = ridge_grid.best_params_['alpha']
best_ridge_model = ridge_grid.best_estimator_

print(f"Best Ridge alpha: {best_ridge_alpha}")
print(f"Best CV MSE: {-ridge_grid.best_score_:.6f}")

Best Ridge alpha: 10
Best CV MSE: 0.056557


In [94]:
# Ridge CV results
ridge_cv_results = pd.DataFrame({
    'Alpha': ridge_alphas,
    'Mean CV MSE': -ridge_grid.cv_results_['mean_test_score'],
    'Std CV MSE': ridge_grid.cv_results_['std_test_score']
})

print(ridge_cv_results.to_string(index=False))

  Alpha  Mean CV MSE  Std CV MSE
  0.001     0.056567    0.002953
  0.010     0.056567    0.002953
  0.100     0.056567    0.002954
  1.000     0.056565    0.002961
 10.000     0.056557    0.003013
100.000     0.056581    0.003208


In [95]:
# Ridge test set evaluation
y_ridge_pred_proba = best_ridge_model.predict(X_test_scaled)
y_ridge_pred_proba = np.clip(y_ridge_pred_proba, 0, 1)
y_ridge_pred_class = (y_ridge_pred_proba > 0.5).astype(int)

ridge_mse = mean_squared_error(y_test, y_ridge_pred_proba)
ridge_r2 = r2_score(y_test, y_ridge_pred_proba)
ridge_accuracy = accuracy_score(y_test, y_ridge_pred_class)
ridge_precision = precision_score(y_test, y_ridge_pred_class, zero_division=0)
ridge_recall = recall_score(y_test, y_ridge_pred_class, zero_division=0)
ridge_f1 = f1_score(y_test, y_ridge_pred_class, zero_division=0)
ridge_roc_auc = roc_auc_score(y_test, y_ridge_pred_proba)
ridge_cm = confusion_matrix(y_test, y_ridge_pred_class)

print(f"MSE: {ridge_mse:.6f}")
print(f"R2: {ridge_r2:.6f}")
print(f"Accuracy: {ridge_accuracy:.4f}")
print(f"Precision: {ridge_precision:.4f}")
print(f"Recall: {ridge_recall:.4f}")
print(f"F1: {ridge_f1:.4f}")
print(f"ROC AUC: {ridge_roc_auc:.4f}")
print(f"Confusion matrix:\n{ridge_cm}")

coef_df_ridge = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": best_ridge_model.coef_
})
coef_df_ridge["Abs_Coefficient"] = coef_df_ridge["Coefficient"].abs()
coef_df_ridge = coef_df_ridge.sort_values("Abs_Coefficient", ascending=False)

print(coef_df_ridge.head(10)[['Feature', 'Coefficient']].to_string(index=False))

MSE: 0.053965
R2: 0.378167
Accuracy: 0.9290
Precision: 0.9032
Recall: 0.2917
F1: 0.4409
ROC AUC: 0.9424
Confusion matrix:
[[901   3]
 [ 68  28]]
                Feature  Coefficient
Income / Median in city     0.116548
             CD.Account     0.088649
              Education     0.062395
             Experience     0.052503
                    Age    -0.051734
 Median Income Per City     0.049635
                  CCAvg     0.041459
                 Family     0.033711
             CreditCard    -0.021830
     Securities.Account    -0.021005


In [96]:
# Ridge training set evaluation
y_ridge_train_pred_proba = best_ridge_model.predict(X_train_scaled)
y_ridge_train_pred_proba = np.clip(y_ridge_train_pred_proba, 0, 1)
y_ridge_train_pred_class = (y_ridge_train_pred_proba > 0.5).astype(int)

ridge_train_mse = mean_squared_error(y_train, y_ridge_train_pred_proba)
ridge_train_r2 = r2_score(y_train, y_ridge_train_pred_proba)
ridge_train_accuracy = accuracy_score(y_train, y_ridge_train_pred_class)
ridge_train_precision = precision_score(y_train, y_ridge_train_pred_class, zero_division=0)
ridge_train_recall = recall_score(y_train, y_ridge_train_pred_class, zero_division=0)
ridge_train_f1 = f1_score(y_train, y_ridge_train_pred_class, zero_division=0)
ridge_train_roc_auc = roc_auc_score(y_train, y_ridge_train_pred_proba)

print(f"Train MSE: {ridge_train_mse:.6f}")
print(f"Train R2: {ridge_train_r2:.6f}")
print(f"Train Accuracy: {ridge_train_accuracy:.4f}")
print(f"Train Precision: {ridge_train_precision:.4f}")
print(f"Train Recall: {ridge_train_recall:.4f}")
print(f"Train F1: {ridge_train_f1:.4f}")
print(f"Train ROC AUC: {ridge_train_roc_auc:.4f}")

print(f"Test-Train MSE diff: {ridge_mse - ridge_train_mse:+.6f}")
print(f"Train-Test R2 diff: {ridge_train_r2 - ridge_r2:+.6f}")
print(f"Train-Test Accuracy diff: {ridge_train_accuracy - ridge_accuracy:+.4f}")
print(f"Train-Test F1 diff: {ridge_train_f1 - ridge_f1:+.4f}")
if ridge_mse > ridge_train_mse:
    print("Test MSE > Train MSE (some overfitting)")
else:
    print("Test MSE <= Train MSE")

Train MSE: 0.053251
Train R2: 0.386392
Train Accuracy: 0.9290
Train Precision: 0.8425
Train Recall: 0.3203
Train F1: 0.4642
Train ROC AUC: 0.9437
Test-Train MSE diff: +0.000714
Train-Test R2 diff: +0.008225
Train-Test Accuracy diff: +0.0000
Train-Test F1 diff: +0.0232
Test MSE > Train MSE (some overfitting)


In [97]:

print("LASSO REGRESSION - HYPERPARAMETER TUNING")

# Define alpha grid - STRONGER to force feature selection
lasso_alphas = [0.01, 0.1, 1, 10, 100]

# Perform GridSearchCV on training set
lasso = Lasso(max_iter=10000)
lasso_grid = GridSearchCV(
    lasso,
    param_grid={'alpha': lasso_alphas},
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

lasso_grid.fit(X_train_scaled, y_train)
best_lasso_alpha = lasso_grid.best_params_['alpha']
best_lasso_model = lasso_grid.best_estimator_

print(f"\nBest Lasso alpha (from CV): {best_lasso_alpha}")
print(f"Best CV MSE: {-lasso_grid.best_score_:.6f}")

LASSO REGRESSION - HYPERPARAMETER TUNING

Best Lasso alpha (from CV): 0.01
Best CV MSE: 0.057746


In [98]:
# Lasso Cross-Validation Results
lasso_cv_results = pd.DataFrame({
    'Alpha': lasso_alphas,
    'Mean CV MSE': -lasso_grid.cv_results_['mean_test_score'],
    'Std CV MSE': lasso_grid.cv_results_['std_test_score']
})

print(lasso_cv_results.to_string(index=False))
min_mse_idx = lasso_cv_results['Mean CV MSE'].idxmin()
max_mse_idx = lasso_cv_results['Mean CV MSE'].idxmax()
mse_improvement = lasso_cv_results.loc[max_mse_idx, 'Mean CV MSE'] - lasso_cv_results.loc[min_mse_idx, 'Mean CV MSE']
improvement_pct = (mse_improvement / lasso_cv_results.loc[max_mse_idx, 'Mean CV MSE']) * 100
print(f"\nBest Alpha: {best_lasso_alpha}")
print(f"Best CV MSE: {-lasso_grid.best_score_:.6f}")
print(f"Std Dev: {lasso_cv_results[lasso_cv_results['Alpha']==best_lasso_alpha]['Std CV MSE'].values[0]:.6f}")
print(f"Worst Alpha: {lasso_cv_results.loc[max_mse_idx, 'Alpha']:.3f} (MSE: {lasso_cv_results.loc[max_mse_idx, 'Mean CV MSE']:.6f})")
print(f"Best Alpha:  {lasso_cv_results.loc[min_mse_idx, 'Alpha']:.3f} (MSE: {lasso_cv_results.loc[min_mse_idx, 'Mean CV MSE']:.6f})")
print(f"MSE Improvement: {mse_improvement:.6f} ({improvement_pct:.2f}%)")

 Alpha  Mean CV MSE  Std CV MSE
  0.01     0.057746    0.003793
  0.10     0.082862    0.008285
  1.00     0.086840    0.008078
 10.00     0.086840    0.008078
100.00     0.086840    0.008078

Best Alpha: 0.01
Best CV MSE: 0.057746
Std Dev: 0.003793
Worst Alpha: 1.000 (MSE: 0.086840)
Best Alpha:  0.010 (MSE: 0.057746)
MSE Improvement: 0.029094 (33.50%)


In [99]:
# Evaluate on TEST SET
y_lasso_pred_proba = best_lasso_model.predict(X_test_scaled)
y_lasso_pred_proba = np.clip(y_lasso_pred_proba, 0, 1)
y_lasso_pred_class = (y_lasso_pred_proba > 0.5).astype(int)

# Calculate metrics
lasso_mse = mean_squared_error(y_test, y_lasso_pred_proba)
lasso_r2 = r2_score(y_test, y_lasso_pred_proba)
lasso_accuracy = accuracy_score(y_test, y_lasso_pred_class)
lasso_precision = precision_score(y_test, y_lasso_pred_class, zero_division=0)
lasso_recall = recall_score(y_test, y_lasso_pred_class, zero_division=0)
lasso_f1 = f1_score(y_test, y_lasso_pred_class, zero_division=0)
lasso_roc_auc = roc_auc_score(y_test, y_lasso_pred_proba)
lasso_cm = confusion_matrix(y_test, y_lasso_pred_class)
n_nonzero_lasso = np.sum(best_lasso_model.coef_ != 0)

print('Lasso MSE:', lasso_mse)
print('Lasso R2:', lasso_r2)
print('Lasso Accuracy:', lasso_accuracy)
print('Lasso Precision:', lasso_precision)
print('Lasso Recall:', lasso_recall)
print('Lasso F1:', lasso_f1)
print('Lasso ROC AUC:', lasso_roc_auc)
print('Lasso Confusion Matrix:\n', lasso_cm)

coef_df_lasso = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": best_lasso_model.coef_
})
coef_df_lasso["Abs_Coefficient"] = coef_df_lasso["Coefficient"].abs()
coef_df_lasso = coef_df_lasso.sort_values("Abs_Coefficient", ascending=False)

print('Top 12 Lasso Coefficients:')
print(coef_df_lasso.head(12)[['Feature', 'Coefficient']])

Lasso MSE: 0.056599209241801035
Lasso R2: 0.3478151589947336
Lasso Accuracy: 0.927
Lasso Precision: 0.96
Lasso Recall: 0.25
Lasso F1: 0.39669421487603307
Lasso ROC AUC: 0.9362440081120944
Lasso Confusion Matrix:
 [[903   1]
 [ 72  24]]
Top 12 Lasso Coefficients:
                    Feature  Coefficient
11  Income / Median in city     0.101928
7                CD.Account     0.070888
4                 Education     0.049988
3                     CCAvg     0.039857
10   Median Income Per City     0.034404
2                    Family     0.021906
9                CreditCard    -0.006709
5                  Mortgage     0.004998
6        Securities.Account    -0.004861
0                       Age    -0.000000
1                Experience    -0.000000
8                    Online    -0.000000


In [100]:
# Evaluate on TRAINING SET for comparison
y_lasso_train_pred_proba = best_lasso_model.predict(X_train_scaled)
y_lasso_train_pred_proba = np.clip(y_lasso_train_pred_proba, 0, 1)
y_lasso_train_pred_class = (y_lasso_train_pred_proba > 0.5).astype(int)

# Calculate training metrics
lasso_train_mse = mean_squared_error(y_train, y_lasso_train_pred_proba)
lasso_train_r2 = r2_score(y_train, y_lasso_train_pred_proba)
lasso_train_accuracy = accuracy_score(y_train, y_lasso_train_pred_class)
lasso_train_precision = precision_score(y_train, y_lasso_train_pred_class, zero_division=0)
lasso_train_recall = recall_score(y_train, y_lasso_train_pred_class, zero_division=0)
lasso_train_f1 = f1_score(y_train, y_lasso_train_pred_class, zero_division=0)
lasso_train_roc_auc = roc_auc_score(y_train, y_lasso_train_pred_proba)

print('Lasso Train MSE:', lasso_train_mse)
print('Lasso Train R2:', lasso_train_r2)
print('Lasso Train Accuracy:', lasso_train_accuracy)
print('Lasso Train Precision:', lasso_train_precision)
print('Lasso Train Recall:', lasso_train_recall)
print('Lasso Train F1:', lasso_train_f1)
print('Lasso Train ROC AUC:', lasso_train_roc_auc)

Lasso Train MSE: 0.0559276761820594
Lasso Train R2: 0.35555314133873284
Lasso Train Accuracy: 0.924
Lasso Train Precision: 0.8703703703703703
Lasso Train Recall: 0.24479166666666666
Lasso Train F1: 0.3821138211382114
Lasso Train ROC AUC: 0.9390757512905604


In [101]:
# Elastic Net Hyperparameter Tuning
elastic_alphas = [0.01, 0.1, 1, 10]
elastic_l1_ratios = [0.2, 0.5, 0.8]
enet = ElasticNet(max_iter=10000)
enet_grid = GridSearchCV(
    enet,
    param_grid={'alpha': elastic_alphas, 'l1_ratio': elastic_l1_ratios},
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
enet_grid.fit(X_train_scaled, y_train)
best_enet_alpha = enet_grid.best_params_['alpha']
best_enet_l1_ratio = enet_grid.best_params_['l1_ratio']
best_enet_model = enet_grid.best_estimator_
print('Best Elastic Net alpha:', best_enet_alpha)
print('Best Elastic Net l1_ratio:', best_enet_l1_ratio)
print('Best CV MSE:', -enet_grid.best_score_)

Best Elastic Net alpha: 0.01
Best Elastic Net l1_ratio: 0.2
Best CV MSE: 0.05661969638130171


In [102]:
# Elastic Net Cross-Validation Results
enet_cv_results = pd.DataFrame({
    'Alpha': enet_grid.cv_results_['param_alpha'],
    'L1 Ratio': enet_grid.cv_results_['param_l1_ratio'],
    'Mean CV MSE': -enet_grid.cv_results_['mean_test_score'],
    'Std CV MSE': enet_grid.cv_results_['std_test_score']
})
print('Elastic Net CV Results:')
print(enet_cv_results)

Elastic Net CV Results:
    Alpha  L1 Ratio  Mean CV MSE  Std CV MSE
0    0.01       0.2     0.056620    0.003238
1    0.01       0.5     0.056862    0.003446
2    0.01       0.8     0.057338    0.003661
3    0.10       0.2     0.060910    0.005074
4    0.10       0.5     0.069025    0.006664
5    0.10       0.8     0.076737    0.007460
6    1.00       0.2     0.086840    0.008078
7    1.00       0.5     0.086840    0.008078
8    1.00       0.8     0.086840    0.008078
9   10.00       0.2     0.086840    0.008078
10  10.00       0.5     0.086840    0.008078
11  10.00       0.8     0.086840    0.008078


In [103]:
# Evaluate Elastic Net on TEST SET
y_enet_pred_proba = best_enet_model.predict(X_test_scaled)
y_enet_pred_proba = np.clip(y_enet_pred_proba, 0, 1)
y_enet_pred_class = (y_enet_pred_proba > 0.5).astype(int)
enet_mse = mean_squared_error(y_test, y_enet_pred_proba)
enet_r2 = r2_score(y_test, y_enet_pred_proba)
enet_accuracy = accuracy_score(y_test, y_enet_pred_class)
enet_precision = precision_score(y_test, y_enet_pred_class, zero_division=0)
enet_recall = recall_score(y_test, y_enet_pred_class, zero_division=0)
enet_f1 = f1_score(y_test, y_enet_pred_class, zero_division=0)
enet_roc_auc = roc_auc_score(y_test, y_enet_pred_proba)
enet_cm = confusion_matrix(y_test, y_enet_pred_class)
n_selected_enet = np.sum(best_enet_model.coef_ != 0)
print('Elastic Net MSE:', enet_mse)
print('Elastic Net R2:', enet_r2)
print('Elastic Net Accuracy:', enet_accuracy)
print('Elastic Net Precision:', enet_precision)
print('Elastic Net Recall:', enet_recall)
print('Elastic Net F1:', enet_f1)
print('Elastic Net ROC AUC:', enet_roc_auc)
print('Elastic Net Confusion Matrix:\n', enet_cm)
coef_df_enet = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": best_enet_model.coef_
})
coef_df_enet["Abs_Coefficient"] = coef_df_enet["Coefficient"].abs()
coef_df_enet = coef_df_enet.sort_values("Abs_Coefficient", ascending=False)
print('Top 12 Elastic Net Coefficients:')
print(coef_df_enet.head(12)[['Feature', 'Coefficient']])

Elastic Net MSE: 0.054532700256168264
Elastic Net R2: 0.37162725552903453
Elastic Net Accuracy: 0.929
Elastic Net Precision: 0.9310344827586207
Elastic Net Recall: 0.28125
Elastic Net F1: 0.432
Elastic Net ROC AUC: 0.9413371128318584
Elastic Net Confusion Matrix:
 [[902   2]
 [ 69  27]]
Top 12 Elastic Net Coefficients:
                    Feature  Coefficient
11  Income / Median in city     0.112604
7                CD.Account     0.084680
4                 Education     0.058279
10   Median Income Per City     0.045927
3                     CCAvg     0.041306
2                    Family     0.030887
9                CreditCard    -0.018517
6        Securities.Account    -0.017579
5                  Mortgage     0.010071
8                    Online    -0.009676
0                       Age     0.000000
1                Experience     0.000000


In [104]:
# Evaluate Elastic Net on TRAINING SET
y_enet_train_pred_proba = best_enet_model.predict(X_train_scaled)
y_enet_train_pred_proba = np.clip(y_enet_train_pred_proba, 0, 1)
y_enet_train_pred_class = (y_enet_train_pred_proba > 0.5).astype(int)
enet_train_mse = mean_squared_error(y_train, y_enet_train_pred_proba)
enet_train_r2 = r2_score(y_train, y_enet_train_pred_proba)
enet_train_accuracy = accuracy_score(y_train, y_enet_train_pred_class)
enet_train_precision = precision_score(y_train, y_enet_train_pred_class, zero_division=0)
enet_train_recall = recall_score(y_train, y_enet_train_pred_class, zero_division=0)
enet_train_f1 = f1_score(y_train, y_enet_train_pred_class, zero_division=0)
enet_train_roc_auc = roc_auc_score(y_train, y_enet_train_pred_proba)
print('Elastic Net Train MSE:', enet_train_mse)
print('Elastic Net Train R2:', enet_train_r2)
print('Elastic Net Train Accuracy:', enet_train_accuracy)
print('Elastic Net Train Precision:', enet_train_precision)
print('Elastic Net Train Recall:', enet_train_recall)
print('Elastic Net Train F1:', enet_train_f1)
print('Elastic Net Train ROC AUC:', enet_train_roc_auc)

Elastic Net Train MSE: 0.05379291002583374
Elastic Net Train R2: 0.38015175578639215
Elastic Net Train Accuracy: 0.9285
Elastic Net Train Precision: 0.855072463768116
Elastic Net Train Recall: 0.3072916666666667
Elastic Net Train F1: 0.4521072796934866
Elastic Net Train ROC AUC: 0.9432232612002213


In [105]:
# Create comprehensive comparison of all three models

print("COMPREHENSIVE MODEL COMPARISON - TEST SET")


comparison_df = pd.DataFrame({
    'Model': ['Ridge', 'Lasso', 'Elastic Net'],
    'MSE': [ridge_mse, lasso_mse, enet_mse],
    'R²': [ridge_r2, lasso_r2, enet_r2],
    'Accuracy': [ridge_accuracy, lasso_accuracy, enet_accuracy],
    'Precision': [ridge_precision, lasso_precision, enet_precision],
    'Recall': [ridge_recall, lasso_recall, enet_recall],
    'F1-Score': [ridge_f1, lasso_f1, enet_f1],
    'ROC AUC': [ridge_roc_auc, lasso_roc_auc, enet_roc_auc]
})

print("\nTest Set Performance Metrics:")
print(comparison_df.to_string(index=False))


print("COMPREHENSIVE MODEL COMPARISON - TRAINING SET")


comparison_train_df = pd.DataFrame({
    'Model': ['Ridge', 'Lasso', 'Elastic Net'],
    'MSE': [ridge_train_mse, lasso_train_mse, enet_train_mse],
    'R²': [ridge_train_r2, lasso_train_r2, enet_train_r2],
    'Accuracy': [ridge_train_accuracy, lasso_train_accuracy, enet_train_accuracy],
    'Precision': [ridge_train_precision, lasso_train_precision, enet_train_precision],
    'Recall': [ridge_train_recall, lasso_train_recall, enet_train_recall],
    'F1-Score': [ridge_train_f1, lasso_train_f1, enet_train_f1],
    'ROC AUC': [ridge_train_roc_auc, lasso_train_roc_auc, enet_train_roc_auc]
})

print("\nTraining Set Performance Metrics:")
print(comparison_train_df.to_string(index=False))

# Identify best model for each metric
print("\n" + "="*80)
print("BEST PERFORMING MODEL BY METRIC (TEST SET)")
print("="*80)
print(f"Lowest MSE:          {comparison_df.loc[comparison_df['MSE'].idxmin(), 'Model']} ({comparison_df['MSE'].min():.6f})")
print(f"Highest R²:          {comparison_df.loc[comparison_df['R²'].idxmax(), 'Model']} ({comparison_df['R²'].max():.6f})")
print(f"Highest Accuracy:    {comparison_df.loc[comparison_df['Accuracy'].idxmax(), 'Model']} ({comparison_df['Accuracy'].max():.4f})")
print(f"Highest Precision:   {comparison_df.loc[comparison_df['Precision'].idxmax(), 'Model']} ({comparison_df['Precision'].max():.4f})")
print(f"Highest Recall:      {comparison_df.loc[comparison_df['Recall'].idxmax(), 'Model']} ({comparison_df['Recall'].max():.4f})")
print(f"Highest F1-Score:    {comparison_df.loc[comparison_df['F1-Score'].idxmax(), 'Model']} ({comparison_df['F1-Score'].max():.4f})")
print(f"Highest ROC AUC:     {comparison_df.loc[comparison_df['ROC AUC'].idxmax(), 'Model']} ({comparison_df['ROC AUC'].max():.4f})")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"""
All three regularized regression models (Ridge, Lasso, Elastic Net) have been
evaluated on the same 80/20 train/test split with identical preprocessing.

Key Findings:
- Ridge Regression: Shrinks all coefficients, uses all features
- Lasso Regression: Performs feature selection with {n_nonzero_lasso} / {len(X.columns)} non-zero coefficients
- Elastic Net: Combines both approaches with {n_selected_enet} / {len(X.columns)} selected variables

Training vs Test Performance:
- Ridge: Train MSE = {ridge_train_mse:.6f}, Test MSE = {ridge_mse:.6f}
- Lasso: Train MSE = {lasso_train_mse:.6f}, Test MSE = {lasso_mse:.6f}
- Elastic Net: Train MSE = {enet_train_mse:.6f}, Test MSE = {enet_mse:.6f}

These models will be compared against OLS (baseline) and SVM (non-linear) models
to determine which approach best predicts personal loan acceptance.
""")


COMPREHENSIVE MODEL COMPARISON - TEST SET

Test Set Performance Metrics:
      Model      MSE       R²  Accuracy  Precision   Recall  F1-Score  ROC AUC
      Ridge 0.053965 0.378167     0.929   0.903226 0.291667  0.440945 0.942420
      Lasso 0.056599 0.347815     0.927   0.960000 0.250000  0.396694 0.936244
Elastic Net 0.054533 0.371627     0.929   0.931034 0.281250  0.432000 0.941337
COMPREHENSIVE MODEL COMPARISON - TRAINING SET

Training Set Performance Metrics:
      Model      MSE       R²  Accuracy  Precision   Recall  F1-Score  ROC AUC
      Ridge 0.053251 0.386392    0.9290   0.842466 0.320312  0.464151 0.943673
      Lasso 0.055928 0.355553    0.9240   0.870370 0.244792  0.382114 0.939076
Elastic Net 0.053793 0.380152    0.9285   0.855072 0.307292  0.452107 0.943223

BEST PERFORMING MODEL BY METRIC (TEST SET)
Lowest MSE:          Ridge (0.053965)
Highest R²:          Ridge (0.378167)
Highest Accuracy:    Ridge (0.9290)
Highest Precision:   Lasso (0.9600)
Highest Recall:      R